# 03 - Ten-day versus twenty-day windows: resolution or labels?

The ten-day dataset splits the season into 14 windows instead of 7, so it observes the crop
twice as often. It also covers far fewer county-years, because a county only enters it when
every one of its 14 windows was cloud-free enough to be recorded. The headline model uses the
twenty-day data, and this notebook asks why that turns out to be the better trade.

Two effects are separated by training the same plain transformer three ways: on the ten-day
features; on the twenty-day features restricted to the **same** county-years; and on the
twenty-day features with all available labels. The first pair isolates temporal resolution, the
second isolates the number of training labels.

All three arms are scored on the county-years both tables cover, so the comparison is like for
like. The 10-day table is built here from the raw exports, so this notebook adds no
dependency beyond what `00_preprocessing.ipynb` already needs.

In [1]:
# Setup - paths, device, metric helpers. Mirrors 01_headline_register.ipynb.
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

project_folder = Path.cwd()
# walk up until the yield label file is found -> that folder is the project root
while not (project_folder / "Ertragsdaten_USA.csv").exists():
    assert project_folder.parent != project_folder, "repo root not found"
    project_folder = project_folder.parent
ROOT = str(project_folder)
DATA = os.path.join(ROOT, "Claude", "data_cache")
EXPERIMENTS = os.path.join(ROOT, "Claude", "experiments")
CACHE = os.path.join(ROOT, "Claude", "final_results_claude", "cache")

torch.backends.cuda.enable_flash_sdp(False)            # Blackwell math-SDPA workaround
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)
dev = "cuda" if torch.cuda.is_available() else "cpu"


def rmse(prediction, target):
    return float(np.sqrt(np.mean((np.asarray(prediction) - np.asarray(target)) ** 2)))


def mae(prediction, target):
    return float(np.mean(np.abs(np.asarray(prediction) - np.asarray(target))))


def regression_metrics(prediction, target):
    """RMSE, MAE, R2 and normalised RMSE (percent of the mean observed yield)."""
    prediction, target = np.asarray(prediction), np.asarray(target)
    return dict(
        RMSE=rmse(prediction, target),
        MAE=mae(prediction, target),
        R2=float(1 - (np.sum((target - prediction) ** 2) / np.sum((target - target.mean()) ** 2))),
        nRMSE=float(rmse(prediction, target) / target.mean() * 100),
    )


print("ROOT =", ROOT, "| device =", dev)

ROOT = <repository> | device = cuda


## 1. Inputs

In [2]:
# The 20-day table and the evaluation cohort, both produced by 00_preprocessing.ipynb.
N_WINDOWS_20D = 7

county_year_features = pd.read_parquet(DATA + r"\features_20d_leakfree.parquet")
county_year_features["county_id"] = county_year_features.county_id.astype(str)
headline_cohort_ids = set(open(os.path.join(CACHE, "cohort_10d_2025.txt")).read().split())

yield_history_5yr = county_year_features["yield_5yr_avg"].to_numpy(np.float32)
yield_actual = county_year_features["yield"].to_numpy(np.float32)
year = county_year_features.year.to_numpy()
county_id = county_year_features.county_id.to_numpy()

is_train_row = (year != 2025) & (year != 2023)
is_val_row = year == 2023
is_test_row = year == 2025
yield_history_5yr = np.where(np.isnan(yield_history_5yr),
                             np.nanmedian(yield_history_5yr[is_train_row]),
                             yield_history_5yr)

# full 104-statistic tokens (the plain configuration keeps every band statistic)
statistic_suffixes = sorted({c[4:] for c in county_year_features.columns if c.startswith("w00_")})
full_tokens_raw = np.stack(
    [county_year_features[[f"w{w:02d}_{suffix}" for suffix in statistic_suffixes]]
     .to_numpy(np.float32) for w in range(N_WINDOWS_20D)], axis=1)
token_mean_train = np.nanmean(full_tokens_raw[is_train_row], axis=(0, 1))
token_std_train = np.nanstd(full_tokens_raw[is_train_row], axis=(0, 1)) + 1e-6
full_tokens_20d = np.nan_to_num((full_tokens_raw - token_mean_train) / token_std_train)

yield_actual_test = yield_actual[is_test_row]
is_headline_cohort = np.array([c in headline_cohort_ids for c in county_id[is_test_row]])
print(f"20-day tokens {full_tokens_20d.shape} | train {is_train_row.sum()}"
      f" | test(2025) {is_test_row.sum()} (headline cohort {is_headline_cohort.sum()})")

20-day tokens (4120, 7, 104) | train 3181 | test(2025) 426 (headline cohort 282)


In [3]:
# The 10-day table: 14 windows instead of 7, but far fewer county-years, because a county only
# enters it when every one of its 14 windows was recorded. It is built here from the raw exports
# with exactly the flattening and leak-free history logic that 00_preprocessing.ipynb applies to
# the 20-day data, so this notebook depends on no file that notebook does not itself produce.
N_WINDOWS_10D = 14
YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2025]
HISTORY_LENGTH = 5
NON_STATISTIC_COLUMNS = {"date_start", "date_end", "from_data"}
FEATURES_10D_FILE = os.path.join(DATA, "features_10d_leakfree.parquet")


def build_10d_feature_table():
    """Flatten the raw 10-day exports into one row per county-year, with leak-free history."""
    import glob
    yield_table = pd.read_csv(os.path.join(ROOT, "Ertragsdaten_USA.csv"))
    yield_columns = {int(c.split()[1]): c for c in yield_table.columns if c.startswith("Yield ")}
    yields_by_county = {}
    for _, row in yield_table.iterrows():
        yields_by_county[str(row["id_combo"])] = {
            year_number: float(row[column]) for year_number, column in yield_columns.items()
            if pd.notna(row[column])}

    rows = []
    statistic_columns = None
    for year_number in YEARS:
        pattern = os.path.join(ROOT, f"Data_Windowed_{year_number}_10d", "Harvester_*.csv")
        for file_path in glob.glob(pattern):
            name_parts = os.path.basename(file_path)[:-4].split("_")
            county = f"{name_parts[2]}_{name_parts[3]}"
            window_table = pd.read_csv(file_path)
            if len(window_table) != N_WINDOWS_10D:
                continue                                   # incomplete season
            if statistic_columns is None:
                statistic_columns = [c for c in window_table.columns
                                     if c not in NON_STATISTIC_COLUMNS]
            county_yields = yields_by_county.get(county, {})
            if year_number not in county_yields:
                continue                                   # no yield label
            prior_year_yields = [county_yields[year_number - k]
                                 for k in range(1, HISTORY_LENGTH + 1)
                                 if (year_number - k) in county_yields]
            record = {"county_id": county, "year": year_number,
                      "yield": county_yields[year_number],
                      "yield_5yr_avg": float(np.mean(prior_year_yields))
                      if prior_year_yields else np.nan}
            window_values = window_table[statistic_columns].to_numpy(np.float32)
            for window in range(N_WINDOWS_10D):
                for position, column in enumerate(statistic_columns):
                    record[f"w{window:02d}_{column}"] = window_values[window, position]
            rows.append(record)
        print(f"  {year_number}: {len(rows)} rows so far", flush=True)
    table = pd.DataFrame(rows)
    table.to_parquet(FEATURES_10D_FILE, index=False)
    return table


if os.path.exists(FEATURES_10D_FILE):
    features_10d = pd.read_parquet(FEATURES_10D_FILE)
    print("loaded cached", os.path.basename(FEATURES_10D_FILE))
else:
    print("building the 10-day feature table from the raw exports (~3 min) ...", flush=True)
    features_10d = build_10d_feature_table()
features_10d["county_id"] = features_10d.county_id.astype(str)
print(f"10-day table: {features_10d.shape[0]} county-years x {features_10d.shape[1]} columns")

statistic_suffixes_10d = sorted({c[4:] for c in features_10d.columns if c.startswith("w00_")})
tokens_10d_raw = np.stack(
    [features_10d[[f"w{w:02d}_{suffix}" for suffix in statistic_suffixes_10d]]
     .to_numpy(np.float32) for w in range(N_WINDOWS_10D)], axis=1)

yield_history_10d = features_10d["yield_5yr_avg"].to_numpy(np.float32)
yield_actual_10d = features_10d["yield"].to_numpy(np.float32)
year_10d = features_10d.year.to_numpy()
county_id_10d = features_10d.county_id.to_numpy()

is_train_row_10d = (year_10d != 2025) & (year_10d != 2023)
is_val_row_10d = year_10d == 2023
is_test_row_10d = year_10d == 2025
yield_history_10d = np.where(np.isnan(yield_history_10d),
                             np.nanmedian(yield_history_10d[is_train_row_10d]),
                             yield_history_10d)
token_mean_train_10d = np.nanmean(tokens_10d_raw[is_train_row_10d], axis=(0, 1))
token_std_train_10d = np.nanstd(tokens_10d_raw[is_train_row_10d], axis=(0, 1)) + 1e-6
tokens_10d = np.nan_to_num((tokens_10d_raw - token_mean_train_10d) / token_std_train_10d)

# county-years covered by BOTH tables -> the only fair ground for a 10-day vs 20-day comparison
county_years_10d = set(zip(features_10d.county_id, features_10d.year))
shared_test_counties = set(county_id_10d[is_test_row_10d]) & set(county_id[is_test_row])
is_shared_10d = np.array([c in shared_test_counties for c in county_id_10d[is_test_row_10d]])
is_shared_20d = np.array([c in shared_test_counties for c in county_id[is_test_row]])
print(f"10-day tokens {tokens_10d.shape} | train {is_train_row_10d.sum()}"
      f" | 2025 rows {is_test_row_10d.sum()}")
print(f"county-years in both tables (comparison set): {len(shared_test_counties)}")

building the 10-day feature table from the raw exports (~3 min) ...
  2016: 252 rows so far
  2017: 594 rows so far
  2018: 911 rows so far
  2019: 1317 rows so far
  2020: 1466 rows so far
  2021: 1542 rows so far
  2022: 1763 rows so far
  2023: 2049 rows so far
  2025: 2331 rows so far
10-day table: 2331 county-years x 1460 columns
10-day tokens (2331, 14, 104) | train 1763 | 2025 rows 282
county-years in both tables (comparison set): 282


## 2. The plain window transformer
One architecture for every arm, so that only the data differs between them.

In [4]:
# The plain window transformer: per-window tokens -> transformer encoder -> attention pooling
# -> fused with the 5-year yield history -> one prediction. No cross-attention block, no soil,
# no coverage bias. This is the configuration the headline model is compared against.
MODEL_DIM = 64


class PlainWindowTransformer(nn.Module):
    def __init__(self, features_per_window, n_windows, d=MODEL_DIM):
        super().__init__()
        self.proj = nn.Linear(features_per_window, d)
        self.pos = nn.Parameter(torch.zeros(n_windows, d))
        self.enc = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, 4, 2 * d, 0.1, activation="gelu",
                                       batch_first=True, norm_first=True), 1)
        self.attn = nn.Linear(d, 1)
        self.stat = nn.Sequential(nn.Linear(1, d), nn.GELU())
        self.head = nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(0.1), nn.Linear(d, 1))

    def forward(self, window_tokens, history):
        encoded = self.enc(self.proj(window_tokens) + self.pos)
        pooling_weights = self.attn(encoded).softmax(1)
        pooled = (encoded * pooling_weights).sum(1)
        return self.head(torch.cat([pooled, self.stat(history)], 1))


def train_plain_transformer(window_tokens, yield_history, yield_actual,
                            is_train_row, is_val_row, is_test_row, n_windows):
    """Train one plain transformer and return its predictions for the test rows."""
    random_generator = np.random.default_rng()
    train_anomalies = yield_actual[is_train_row] - yield_history[is_train_row]
    anomaly_mean_train = float(train_anomalies.mean())
    anomaly_std_train = float(train_anomalies.std())

    def to_gpu_tensor(array, row_mask):
        return torch.tensor(array[row_mask], device=dev)

    tokens_train = to_gpu_tensor(window_tokens, is_train_row)
    history_train = to_gpu_tensor(yield_history[:, None], is_train_row)
    target_train = torch.tensor(
        ((train_anomalies - anomaly_mean_train) / anomaly_std_train)[:, None], device=dev)
    tokens_val = to_gpu_tensor(window_tokens, is_val_row)
    history_val = to_gpu_tensor(yield_history[:, None], is_val_row)
    tokens_test = to_gpu_tensor(window_tokens, is_test_row)
    history_test = to_gpu_tensor(yield_history[:, None], is_test_row)

    model = PlainWindowTransformer(window_tokens.shape[2], n_windows).to(dev)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, 300)
    loss_function = nn.SmoothL1Loss()

    best_validation_rmse = 1e9
    best_state_dict = None
    epochs_without_improvement = 0
    minimum_improvement = 1e-4
    patience = 25

    for epoch in range(300):
        model.train()
        shuffled_rows = random_generator.permutation(len(tokens_train))
        for start in range(0, len(shuffled_rows), 64):
            batch_rows = shuffled_rows[start:start + 64]
            noisy_tokens = tokens_train[batch_rows] + torch.randn_like(tokens_train[batch_rows]) * 0.03
            optimizer.zero_grad()
            batch_loss = loss_function(model(noisy_tokens, history_train[batch_rows]),
                                       target_train[batch_rows])
            batch_loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            standardized_val_prediction = model(tokens_val, history_val)
        val_prediction = (standardized_val_prediction.squeeze(1).cpu().numpy() * anomaly_std_train
                          + anomaly_mean_train + yield_history[is_val_row])
        validation_rmse = rmse(val_prediction, yield_actual[is_val_row])

        if validation_rmse < best_validation_rmse - minimum_improvement:
            best_validation_rmse = validation_rmse
            best_state_dict = {name: value.detach().cpu().clone()
                               for name, value in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state_dict)
    model.eval()
    with torch.no_grad():
        standardized_test_prediction = model(tokens_test, history_test)
    return (standardized_test_prediction.squeeze(1).cpu().numpy() * anomaly_std_train
            + anomaly_mean_train + yield_history[is_test_row])


def load_or_train_bank(folder_name, n_models, train_one_model):
    """Return an (n_models, n_test) prediction array, training whatever is not cached."""
    folder = os.path.join(CACHE, folder_name)
    os.makedirs(folder, exist_ok=True)
    predictions = []
    for model_number in range(n_models):
        cache_file = os.path.join(folder, f"model_{model_number}.npy")
        if os.path.exists(cache_file):
            predictions.append(np.load(cache_file))
        else:
            prediction = train_one_model()
            np.save(cache_file, prediction)
            predictions.append(prediction)
            if (model_number + 1) % 10 == 0:
                print(f"  {folder_name}: {model_number + 1}/{n_models} trained", flush=True)
    return np.array(predictions)

## 3. Resolution versus labels

In [5]:
# Three arms, all scored on the shared comparison set:
#   A  10-day features, 14 windows, trained on the county-years the 10-day table covers
#   B  20-day features, training restricted to those SAME county-years  -> isolates resolution
#   C  20-day features, all available county-years                      -> adds the extra labels
# A vs B answers "does finer temporal resolution help?"; B vs C answers "do more labels help?".
N_MODELS = 30

is_train_row_restricted = is_train_row & np.array(
    [(c, y) in county_years_10d for c, y in zip(county_id, year)])
print(f"training rows: 10-day {is_train_row_10d.sum()} | 20-day restricted "
      f"{is_train_row_restricted.sum()} | 20-day all {is_train_row.sum()}")

bank_10d = load_or_train_bank(
    "track_10d_models", N_MODELS,
    lambda: train_plain_transformer(tokens_10d, yield_history_10d, yield_actual_10d,
                                    is_train_row_10d, is_val_row_10d, is_test_row_10d,
                                    N_WINDOWS_10D))
bank_20d_restricted = load_or_train_bank(
    "track_20d_restricted_models", N_MODELS,
    lambda: train_plain_transformer(full_tokens_20d, yield_history_5yr, yield_actual,
                                    is_train_row_restricted, is_val_row, is_test_row,
                                    N_WINDOWS_20D))
bank_20d_all_labels = load_or_train_bank(
    "track_20d_all_labels_models", N_MODELS,
    lambda: train_plain_transformer(full_tokens_20d, yield_history_5yr, yield_actual,
                                    is_train_row, is_val_row, is_test_row, N_WINDOWS_20D))

actual_shared = yield_actual_10d[is_test_row_10d][is_shared_10d]
metrics_10d = regression_metrics(bank_10d.mean(axis=0)[is_shared_10d], actual_shared)
metrics_20d_restricted = regression_metrics(
    bank_20d_restricted.mean(axis=0)[is_shared_20d], actual_shared)
metrics_20d_all = regression_metrics(bank_20d_all_labels.mean(axis=0)[is_shared_20d], actual_shared)

attribution_table = pd.DataFrame([
    {"arm": "A  10-day features, 10-day labels", "training rows": int(is_train_row_10d.sum()),
     **{k: round(v, 3) for k, v in metrics_10d.items()}},
    {"arm": "B  20-day features, same labels", "training rows": int(is_train_row_restricted.sum()),
     **{k: round(v, 3) for k, v in metrics_20d_restricted.items()}},
    {"arm": "C  20-day features, all labels", "training rows": int(is_train_row.sum()),
     **{k: round(v, 3) for k, v in metrics_20d_all.items()}},
])
display(attribution_table)

resolution_effect = metrics_10d["RMSE"] - metrics_20d_restricted["RMSE"]
label_effect = metrics_20d_restricted["RMSE"] - metrics_20d_all["RMSE"]
print(f"resolution effect (A - B, positive = coarser 20-day is better): {resolution_effect:+.3f}")
print(f"label effect      (B - C, positive = more labels help)        : {label_effect:+.3f}")

training rows: 10-day 1763 | 20-day restricted 1763 | 20-day all 3181


<home>\AppData\Local\Temp\ipykernel_30892\3980582915.py:12: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(


KeyboardInterrupt: 

## Reading

The two effects printed above are what the choice of dataset comes down to.

A near-zero **resolution effect** means the finer ten-day sampling buys nothing once the season
is already described by seven windows - wheat develops slowly enough that a twenty-day cadence
captures its trajectory. The **label effect** says whether the extra county-years the twenty-day
table brings are worth having.

Both should be read against the ensemble noise of a few hundredths of a dt/ha; differences
smaller than that are not evidence of anything. Note also that the extra county-years are not a
random sample - they are precisely the counties whose ten-day coverage was incomplete, which are
harder to predict, so they make the training set larger and noisier at the same time.